# 20. Nested Subgraph Multi-Agent Architecture
**Industry:** Insurance

Build a LangGraph system where a top-level supervisor graph calls a nested subgraph (its own internal multi-step workflow) as a single node.

In [ ]:
!pip install langgraph langchain langchain-google-genai

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class ClaimState(TypedDict):
    claim_id: str
    status: str
    damage_cost: int

# --- Inner Graph: Damage Assessment ---
def research_photos(state: ClaimState):
    return {"status": "photos_researched"}

def estimate_cost(state: ClaimState):
    return {"damage_cost": 500}

inner_graph = StateGraph(ClaimState)
inner_graph.add_node("research", research_photos)
inner_graph.add_node("estimate", estimate_cost)
inner_graph.add_edge(START, "research")
inner_graph.add_edge("research", "estimate")
inner_graph.add_edge("estimate", END)
inner_app = inner_graph.compile()

# --- Outer Graph: Claims Supervisor ---
def supervisor_node(state: ClaimState):
    return {"status": "assessing_damage"}

outer_graph = StateGraph(ClaimState)
outer_graph.add_node("supervisor", supervisor_node)
outer_graph.add_node("damage_assessment", inner_app)
outer_graph.add_edge(START, "supervisor")
outer_graph.add_edge("supervisor", "damage_assessment")
outer_graph.add_edge("damage_assessment", END)
outer_app = outer_graph.compile()

for event in outer_app.stream({"claim_id": "C123", "status": "new", "damage_cost": 0}):
    print(event)